# Plots for RNAset

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py
from KDEpy import FFTKDE
from glob import glob

%matplotlib inline


In [ ]:
# ---- configure general ----------------------------------------------------
DATA_ROOT = "mdrnacat"                  # contains the per-system replicas file
H5_NAME   = "{}_replica.h5"     # per-system replicas file
SAVE_DIR  = ""                  # save directory for plots

PDB_IDS = [os.path.basename(f)[:-11] for f in sorted(glob(DATA_ROOT + '/*_replica.h5'))]

def panel_label(ax, letter):
    ax.text(0.0, 1.02, letter, transform=ax.transAxes,
            fontweight="bold", fontsize=13, ha="left", va="bottom")


In [ ]:
# ---- colors ----------------------------------------------------
from matplotlib.colors import LinearSegmentedColormap


CUSTOM_COLORS = {
    "customblue":        "#005AA0",  # RGB(0, 90, 160)
    "customdarkblue":    "#002864",  # RGB(0, 40, 100)
    "customlightblue":   "#14C8FF",  # RGB(20, 200, 255)
    "customgreen":       "#8CB423",  # RGB(140, 180, 35)
    "customgray":        "#5A696E",  # RGB(90, 105, 110)
    "customhighlight":   "#CDEEFB",  # RGB(205, 238, 251)
    "custommint":        "#05E5BA",  # RGB(5, 229, 186)
    "custompale":        "#ECFBFD",  # RGB(236, 251, 253)
    "customaerospace":   "#50C8AA",  # RGB(80, 200, 170)
    "customearth":       "#326469",  # RGB(50, 100, 105)
    "customenergy":      "#FFD228",  # RGB(255, 210, 40)
    "customhealth":      "#D23264",  # RGB(210, 50, 100)
    "custominformation": "#A0235A",  # RGB(160, 35, 90)
    "custommatter":      "#F0781E",  # RGB(240, 120, 30)
}
cmap_fel = LinearSegmentedColormap.from_list('my_cmap', [CUSTOM_COLORS["customdarkblue"],CUSTOM_COLORS["customhealth"],CUSTOM_COLORS["customlightblue"] ])
cmap_alt = LinearSegmentedColormap.from_list('my_cmap', [CUSTOM_COLORS["customenergy"],CUSTOM_COLORS["custommatter"],CUSTOM_COLORS["customearth"] ])
cmap_plain = LinearSegmentedColormap.from_list('my_cmap', [CUSTOM_COLORS["customenergy"],CUSTOM_COLORS["customenergy"] ])
cmap_arrow = LinearSegmentedColormap.from_list('my_cmap', ["#712A40","white" ])
cmap = LinearSegmentedColormap.from_list('my_cmap', [CUSTOM_COLORS["customlightblue"],CUSTOM_COLORS["customhealth"] ])

## Replica-exchange inspection 

In [ ]:
# ---- configure -------------------------------------------------------------

R         = 8.3145        # J K^-1 mol^-1
GRIDSIZE  = 10_000        # KDE evaluation grid points
COMP_ENERGY_LABEL = (r'$E_{RNA-RNA} + \frac{1}{2}'
                     r'\sqrt{\beta_{state}/\beta_0}\, E_{RNA-solvent}$')

PDB_LIST_REPEX = ['3r4f']

In [ ]:
def load_repex(pdbid):
    # Read the raw repex/ parameters from a system's replicas h5.
    path = os.path.join(DATA_ROOT, pdbid, H5_NAME.format(pdbid))
    with h5py.File(path, "r") as f:
        dom = f[list(f.keys())[0]]
        if "repex" not in dom:
            raise KeyError(f"{path}: no repex/ group (build with --repex-raw)")
        rg = dom["repex"]
        terms = [t.decode() if isinstance(t, bytes) else t for t in rg["terms"][:]]
        return dict(
            terms = terms,
            energy = rg["energy"][:],            # (N, S, K)  kJ/mol
            nsamp  = rg["energyNSamples"][:],    # (N,)  valid samples per state
            etime  = rg["energyTime"][:],        # (N, S)  ps
            temps  = rg["temps"][:],             # (N,)  effective ladder temps
            ri     = rg["replicaIndex"][:],      # (X, N)  replica (1-based) at each state
            rt     = rg["replicaIndexTime"][:],  # (X,)  ps
            natt   = rg["nAttempts"][:],         # (P,)
            nexch  = rg["nExchanges"][:],        # (P, N-1)
        )


def comparison_energy_per_state(d):
    # Comparison-energy samples per thermodynamic state:
    #   E_pp + 0.5 * sqrt(beta_max / beta_state) * E_pw
    # with E_pp = sum of RNA-RNA terms, E_pw = sum of RNA-Water terms.
    epp = [i for i, t in enumerate(d["terms"]) if "RNA-RNA" in t]
    epw = [i for i, t in enumerate(d["terms"]) if "RNA-Water" in t]
    beta = 1.0 / (R * d["temps"])
    beta_max = beta.max()
    out = []
    for i in range(d["energy"].shape[0]):
        s = int(d["nsamp"][i])
        e = d["energy"][i, :s, :]
        scale = 0.5 * np.sqrt(beta_max / beta[i])
        out.append(e[:, epp].sum(1) + scale * e[:, epw].sum(1))
    return out


def comparison_energy_per_replica(d, ce_state):
    # Re-bin each per-state comparison-energy sample onto the replica that
    # occupied that state at the sample time (from the demux occupancy).
    n = len(ce_state)
    per_rep = [[] for _ in range(n)]              # 0-based replica index
    rt = d["rt"]
    for i in range(n):
        s = int(d["nsamp"][i])
        rows = np.clip(np.searchsorted(rt, d["etime"][i, :s], side="right") - 1,
                       0, len(rt) - 1)
        reps = d["ri"][rows, i] - 1               # 1-based -> 0-based
        for r, val in zip(reps, ce_state[i]):
            per_rep[int(r)].append(val)
    return [np.asarray(x) for x in per_rep]


def kde_on_grid(values, grid):
    est = FFTKDE(kernel="gaussian", bw="silverman").fit(np.asarray(values)).evaluate(grid)
    return est / est.sum()


def bhattacharyya(p, q):
    return np.sum(np.sqrt(p * q))

def plot_repex(pdbid):
    d = load_repex(pdbid)
    ce_state = comparison_energy_per_state(d)
    ce_rep   = comparison_energy_per_replica(d, ce_state)
    n = len(ce_state)
    states = np.arange(1, n + 1)
    pairs  = np.arange(1, n)                       # adjacent state pairs i/(i+1)

    # shared KDE grid over the full comparison-energy range
    allce = np.concatenate(ce_state)
    lo, hi = allce.min(), allce.max()
    grid = np.linspace(lo - abs(lo) * 1e-4, hi + abs(hi) * 1e-4, GRIDSIZE)
    state_est = np.stack([kde_on_grid(c, grid) for c in ce_state])
    rep_est   = np.stack([kde_on_grid(c, grid) if len(c) else np.zeros_like(grid)
                          for c in ce_rep])

    transition     = d["nexch"].sum(0) / d["natt"].sum()                 # (N-1,)
    energy_overlap = np.array([bhattacharyya(state_est[i], state_est[i + 1])
                               for i in range(n - 1)])                    # (N-1,)

    fig, (ax0, ax1, ax2) = plt.subplots(3, 1, figsize=(12, 8))

    # 1) transition probability (black) + energy overlap (red, twin axis)
    ax0.plot(pairs, transition, '-', color='k')
    ax0.scatter(pairs, transition, c=pairs, cmap=cmap, zorder=3)
    ax0.axhline(np.nanmean(transition), ls='--', color='k', lw=0.8)
    ax0.text(0.05, 0.30, f'ø = {np.nanmean(transition):.2f}', transform=ax0.transAxes, color='k')
    ax0.set_xlabel('Transition'); ax0.set_ylabel('Transition probability'); ax0.set_ylim(0, 1)
    oax = ax0.twinx()
    oax.plot(pairs, energy_overlap, '-s', color=CUSTOM_COLORS["custommatter"], alpha=0.5)
    oax.axhline(np.nanmean(energy_overlap), ls='--', color=CUSTOM_COLORS["custommatter"], lw=0.8)
    oax.text(0.95, 0.65, f'ø = {np.nanmean(energy_overlap):.2f}', transform=oax.transAxes,
             ha='right', color=CUSTOM_COLORS["custommatter"])
    oax.set_ylabel('Energy overlap', color=CUSTOM_COLORS["custommatter"]); oax.tick_params(axis='y', colors=CUSTOM_COLORS["custommatter"]); oax.set_ylim(0, 1)

    # 2) energy density per thermodynamic state (viridis)
    sm_state = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(1, n), cmap=cmap)
    for s_id, est in zip(states, state_est):
        c = sm_state.to_rgba(s_id)
        ax1.plot(grid, est, color=c, lw=0.8); ax1.fill_between(grid, est, color=c, alpha=0.4)
    ax1.set_ylabel('Density'); ax1.set_xlabel(COMP_ENERGY_LABEL)
    fig.colorbar(sm_state, ax=ax1, label='Thermodynamic state')

    # 3) energy density per replica (plasma), lowest dashed black / highest dashed red
    sm_rep = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(1, n), cmap=cmap_alt)
    for s_id, est in zip(states, rep_est):
        c = sm_rep.to_rgba(s_id)
        ax2.plot(grid, est, color=c, lw=0.8); ax2.fill_between(grid, est, color=c, alpha=0.4)
    #ax2.plot(grid, rep_est[0],  ls='-', color=CUSTOM_COLORS["customenergy"], lw=2.0)
    #ax2.plot(grid, rep_est[-1], ls='-', color=CUSTOM_COLORS["customearth"], lw=2.0)
    ax2.set_ylabel('Density'); ax2.set_xlabel(COMP_ENERGY_LABEL)
    fig.colorbar(sm_rep, ax=ax2, label='Replica')

    for ax, lab in zip((ax0, ax1, ax2), "abc"):
        panel_label(ax, lab)
    fig.suptitle(f'{pdbid}', y=1.005)
    plt.tight_layout(); 
    plt.savefig(SAVE_DIR + 'repex.pdf', bbox_inches='tight')
    plt.show()


In [ ]:
for pdbid in PDB_LIST_REPEX:
    try:
        plot_repex(pdbid)
    except (FileNotFoundError, OSError, KeyError) as exc:
        print(f"{pdbid}: skipped ({exc})")

### Set-wide replica-exchange summary


In [ ]:
def set_repex_summary(pdb_ids):
    trans, over = [], []
    for pid in pdb_ids:
        d = load_repex(pid); n = d["energy"].shape[0]
        ce = comparison_energy_per_state(d); allce = np.concatenate(ce)
        lo, hi = allce.min(), allce.max(); pad = 0.01 * (hi - lo)   # grid must enclose all points
        est = np.stack([kde_on_grid(c, np.linspace(lo - pad, hi + pad, GRIDSIZE)) for c in ce])
        trans.append((d["nexch"].sum(0) / d["natt"].sum()).mean())              # mean transition prob
        over.append(np.mean([bhattacharyya(est[i], est[i + 1]) for i in range(n - 1)]))
    trans, over = np.array(trans), np.array(over)
    print(f"Set-wide over {len(pdb_ids)} systems (mean +/- SD across systems):")
    print(f"  transition probability : {trans.mean():.3f} +/- {trans.std():.3f}")
    print(f"  energy overlap         : {over.mean():.3f} +/- {over.std():.3f}")


In [ ]:
set_repex_summary(PDB_IDS)


### Per-system energy overlap grid


In [ ]:
def plot_state_overlap(ax, pdbid):
    # per-state comparison-energy densities for one system (as subfigure b of plot_repex)
    d = load_repex(pdbid)
    ce = comparison_energy_per_state(d); n = len(ce)
    allce = np.concatenate(ce); lo, hi = allce.min(), allce.max(); pad = 0.01 * (hi - lo)
    grid = np.linspace(lo - pad, hi + pad, GRIDSIZE)
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(1, n), cmap=cmap)
    for s_id, c in enumerate(ce, 1):
        est = kde_on_grid(c, grid); col = sm.to_rgba(s_id)
        ax.plot(grid, est, color=col, lw=0.6, rasterized=True)
        ax.fill_between(grid, est, color=col, alpha=0.4, rasterized=True)
    ax.set_title(pdbid); ax.set_yticks([]); ax.tick_params(labelsize=7)
    return sm


def plot_overlap_grid(pdb_ids, nrows=12, ncols=2, tag=""):
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 24),
                             constrained_layout=True)
    sm = None
    for ax, pid in zip(axes.ravel(), pdb_ids):
        sm = plot_state_overlap(ax, pid)
    for ax in axes.ravel()[len(pdb_ids):]:
        ax.axis("off")
    fig.supxlabel(COMP_ENERGY_LABEL); fig.supylabel("Density")
    fig.colorbar(sm, ax=axes, label="Thermodynamic state", shrink=0.15)
    fig.savefig(SAVE_DIR + f"repex_overlap_grid{tag}.pdf", bbox_inches="tight", dpi=150)
    plt.show()


In [ ]:
plot_overlap_grid(PDB_IDS[:24], tag="_1")   # first 24 systems
plot_overlap_grid(PDB_IDS[24:], tag="_2")   # remaining 24 systems


## TICA of the lowest replica

In [ ]:
# ---- configure ----
SINGLE_H5_NAME = "{}.h5"     # lowest-replica (fine) file: coords/forces
TICA_SELECTION = "name P"                # all-atom RNA featurisation (as tica.py 'aa')
TICA_STRIDE    = 10                      # subsample frames (single file is 1 ps -> 10 ps)
TICA_LAG       = 10                      # pyemma tica lag in strided frames (as tica.py)
TICA_BINS      = 45                      # free-energy histogram bins (as tica.py)
NATIVE_PDB = "{}.pdb"     # native/experimental structure to mark on the FEL
N_MINIMA   = 2            # distinct FEL minima to annotate + report
WRITE_MINIMA_PDB = True     # write a PDB of each minimum's representative frame


In [ ]:
import pyemma
import MDAnalysis as mda
from MDAnalysis.analysis.distances import distance_array


def load_lowest_universe(pdbid, stride):
    # lowest replica coords (single/fine file) + onlyrna.gro topology (atom names)
    h5path = os.path.join(DATA_ROOT, pdbid, SINGLE_H5_NAME.format(pdbid))
    top    = os.path.join(DATA_ROOT, pdbid, TOP_NAME.format(pdbid))
    with h5py.File(h5path, "r") as f:
        dom = f[list(f.keys())[0]]
        groups = sorted((k for k in dom if isinstance(dom[k], h5py.Group) and k != "repex"),
                        key=float)                      # lowest effective temperature first
        coords = dom[f"{groups[0]}/0"]["coords"][::stride]   # (T, N, 3) Angstrom
    u = mda.Universe(top)
    u.load_new(coords)
    return u


def pairwise_distances(u, selection):
    # upper-triangle pairwise distances of the selected atoms, per frame (as tica.py)
    sel = u.select_atoms(selection)
    out = []
    for _ in u.trajectory:
        dm = distance_array(sel.positions, sel.positions)
        out.append(dm[np.triu_indices_from(dm, k=1)])
    return np.array(out)


def plot_tica(pdbid):
    # single-replica FEL of one h5: TICA of the lowest replica, shown as a -ln P landscape
    u = load_lowest_universe(pdbid, TICA_STRIDE)
    dist = pairwise_distances(u, TICA_SELECTION)

    tica = pyemma.coordinates.tica(dist, lag=TICA_LAG, dim=2)
    y = tica.get_output()[0]

    hist, xe, ye = np.histogram2d(y[:, 0], y[:, 1], bins=TICA_BINS, density=True)
    zero = np.min(hist[hist != 0])
    hist[hist == 0] = np.nan
    free_energy = -np.log(hist.T) + np.log(zero)               # (ny, nx), kT
    xc = 0.5 * (xe[:-1] + xe[1:]); yc = 0.5 * (ye[:-1] + ye[1:])

    fig, ax = plt.subplots(figsize=(6.5, 6), constrained_layout=True)
    ax.set_box_aspect(1)                                                   # square FEL
    c = ax.contourf(xe[:-1], ye[:-1], free_energy, levels=20, cmap=cmap_fel, extend="min")
    cax = ax.inset_axes([1.03, 0.0, 0.05, 1.0])          # energy bar == FEL height
    fig.colorbar(c, cax=cax, label="Free energy [kT]")
    ax.set_xlabel("TIC 1"); ax.set_ylabel("TIC 2")
    ax.set_title(f"{pdbid}")

    # native structure projected onto the same TICs
    try:
        nat = mda.Universe(os.path.join(DATA_ROOT, pdbid, NATIVE_PDB.format(pdbid)))
        nat_d = pairwise_distances(nat, TICA_SELECTION)
        if nat_d.shape[1] == dist.shape[1]:
            yn = tica.transform(nat_d)[0]
            ax.scatter(yn[0], yn[1], marker="*", s=280, color="white",
                       linewidth=1.2, zorder=6)
            ax.annotate("native", (yn[0], yn[1]), textcoords="offset points",
                        xytext=(-40, 6), fontweight="bold", zorder=6, color="white")
        else:
            print(f"{pdbid}: native P features ({nat_d.shape[1]}) != trajectory "
                  f"({dist.shape[1]}); native marker skipped")
    except (FileNotFoundError, OSError, ValueError) as exc:
        print(f"{pdbid}: native marker skipped ({exc})")

    # N most distinct minima: deepest FEL cells kept >= min_sep bins apart
    valid = np.argwhere(~np.isnan(free_energy))
    order = valid[np.argsort(free_energy[valid[:, 0], valid[:, 1]])]
    min_sep = max(2, TICA_BINS // 6)
    picked = []
    for jy, ix in order:
        if all(np.hypot(jy - pj, ix - pi) >= min_sep for pj, pi in picked):
            picked.append((int(jy), int(ix)))
            if len(picked) == N_MINIMA:
                break
    print(f"{pdbid}: {len(picked)} most distinct FEL minima (lowest replica):")
    for rank, (jy, ix) in enumerate(picked, 1):
        gx, gy = xc[ix], yc[jy]
        fr = int(np.argmin((y[:, 0] - gx) ** 2 + (y[:, 1] - gy) ** 2))
        ax.scatter(gx, gy, marker="o", s=70, facecolor="none",
                   edgecolor="white", linewidth=1.6, zorder=6)
        ax.annotate(str(rank), (gx, gy), textcoords="offset points",
                    xytext=(5, 5), fontweight="bold", zorder=6, color="white")
        print(f"  minimum {rank}: TIC=({gx:.2f}, {gy:.2f})  FE={free_energy[jy, ix]:.2f} kT"
              f"  -> frame {fr} (raw frame {fr * TICA_STRIDE})")

    plt.savefig(SAVE_DIR + 'tica.pdf', bbox_inches='tight')
    plt.show()


In [ ]:
plot_tica('6qn3')

## Replica TICA


In [ ]:
# ---- configure ----
REPLICA_TICA_PANELS = 4     # replicas shown, evenly spaced from lowest to highest
REPLICA_TICA_STRIDE = 1     # replicas file is already coarse (100 ps / 1 ns) -> no decimation
REPLICA_TICA_LAG    = 10    # pyemma tica lag, in replicas-file frames
# reuses TICA_SELECTION, TICA_BINS, TOP_NAME, H5_NAME and pairwise_distances() from above

PDB_IDS_REPLICA = ['2qus', '6cu1', '7elq']

In [ ]:
def _replica_labels(pdbid):
    # ascending effective-temperature group labels in the replicas file
    with h5py.File(os.path.join(DATA_ROOT, pdbid, H5_NAME.format(pdbid)), "r") as f:
        dom = f[list(f.keys())[0]]
        return sorted((k for k in dom if isinstance(dom[k], h5py.Group) and k != "repex"),
                      key=float)


def _replica_universe(pdbid, group, stride):
    top = os.path.join(DATA_ROOT, pdbid, TOP_NAME.format(pdbid))
    with h5py.File(os.path.join(DATA_ROOT, pdbid, H5_NAME.format(pdbid)), "r") as f:
        coords = f[list(f.keys())[0]][f"{group}/0"]["coords"][::stride]
    u = mda.Universe(top); u.load_new(coords)
    return u


def plot_replica_tica_grid(pdb_ids, n_cols=REPLICA_TICA_PANELS):
    # rows = systems, columns = temperatures. TICA is fit on each system's lowest
    # replica and its selected replicas are projected onto those TICs; a row shares
    # one axis (the third replica's), the colour scale is shared across the figure.
    def free_energy(p, xe, ye):
        h, _, _ = np.histogram2d(p[:, 0], p[:, 1], bins=[xe, ye], density=True)
        zero = np.min(h[h > 0]); h[h == 0] = np.nan
        return -np.log(h.T) + np.log(zero)

    rows = []
    for pdbid in pdb_ids:
        try:
            groups = _replica_labels(pdbid)
        except (FileNotFoundError, OSError, KeyError) as exc:
            print(f"{pdbid}: skipped ({exc})"); continue
        idx = np.unique(np.linspace(0, len(groups) - 1, n_cols).round().astype(int))
        low = pairwise_distances(_replica_universe(pdbid, groups[0], REPLICA_TICA_STRIDE),
                                 TICA_SELECTION)
        tica = pyemma.coordinates.tica(low, lag=REPLICA_TICA_LAG, dim=2)
        proj = [tica.transform(pairwise_distances(
                    _replica_universe(pdbid, groups[gi], REPLICA_TICA_STRIDE), TICA_SELECTION))
                for gi in idx]
        ref = proj[2] if len(proj) >= 3 else proj[-1]
        xe = np.linspace(ref[:, 0].min(), ref[:, 0].max(), TICA_BINS + 1)
        ye = np.linspace(ref[:, 1].min(), ref[:, 1].max(), TICA_BINS + 1)
        rows.append(dict(pdbid=pdbid, temps=[float(groups[gi]) for gi in idx],
                         fes=[free_energy(p, xe, ye) for p in proj], xe=xe, ye=ye))
    if not rows:
        print("nothing to plot"); return

    ncols = max(len(r["fes"]) for r in rows)
    nrows = len(rows)
    vmin = min(np.nanmin(fe) for r in rows for fe in r["fes"])

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows),
                             squeeze=False, constrained_layout=True)
    #fig.get_layout_engine().set(rect=(0.1, 0, 0.9, 1))   # left margin for the pdb-id row labels
    for ri, r in enumerate(rows):
        for ci in range(ncols):
            ax = axes[ri][ci]
            if ci >= len(r["fes"]):
                ax.set_axis_off(); continue
            pcm = ax.pcolormesh(r["xe"], r["ye"], np.ma.masked_invalid(r["fes"][ci]),
                                cmap=cmap_fel, vmin=vmin, vmax=0, shading="flat")
            ax.set_xlim(r["xe"][0], r["xe"][-1]); ax.set_ylim(r["ye"][0], r["ye"][-1])
            ax.set_box_aspect(1)                          # square panels
            ax.tick_params(labelleft=(ci == 0))           # y numbers only on leftmost column
            if ri == 0:
                ax.set_title(f"{rows[0]['temps'][ci]:.0f} K")
            if ci == 0:
                ax.annotate(r["pdbid"], xy=(0, 0.5), xycoords="axes fraction",
                            xytext=(-60, 0), textcoords="offset points", rotation=90,
                            ha="center", va="center", annotation_clip=False,
                            fontsize=plt.rcParams["axes.titlesize"])   # row label, styled like temp titles
    fig.supxlabel("TIC 1")
    fig.supylabel("TIC 2",x=0.03)
    fig.colorbar(pcm, ax=axes, label="Free energy [kT]")
    plt.savefig(SAVE_DIR + 'replica_tica.pdf')
    plt.show()

In [ ]:
plot_replica_tica_grid(PDB_IDS_REPLICA)

## Secondary structure & sugar pucker (lowest replica)

In [ ]:
# ---- configure ----
PUCKER_BINS = 36     # wedges around the pseudorotation wheel
# reads the lowest temperature group of H5_NAME; needs the h5 built with --rna-annotations

In [ ]:
_CLOSE = {")": "(", "]": "[", "}": "{", ">": "<"}
_OPEN  = set(_CLOSE.values())


def load_ss_pucker(pdbid):
    # secondaryStructure (F,R dot-bracket) + sugarPucker (F,R degrees) of the lowest replica
    with h5py.File(os.path.join(DATA_ROOT, pdbid, H5_NAME.format(pdbid)), "r") as f:
        dom = f[list(f.keys())[0]]
        groups = sorted((k for k in dom if isinstance(dom[k], h5py.Group) and k != "repex"),
                        key=float)
        g = dom[f"{groups[0]}/0"]
        if "secondaryStructure" not in g:
            raise KeyError("no secondaryStructure (build the h5 with --rna-annotations)")
        ss_raw = g["secondaryStructure"][:]
        pucker = g["sugarPucker"][:]
    ss = np.array([[c.decode() if isinstance(c, bytes) else c for c in row]
                   for row in ss_raw])                       # (F, R) single chars
    return ss, pucker


def _frame_pairs(row):
    # match dot-bracket into (i, j) partner pairs (handles () [] {} <> for pseudoknots)
    stacks = {o: [] for o in _OPEN}
    pairs = []
    for k, ch in enumerate(row):
        if ch in _OPEN:
            stacks[ch].append(k)
        elif ch in _CLOSE and stacks[_CLOSE[ch]]:
            pairs.append((stacks[_CLOSE[ch]].pop(), k))
    return pairs


def base_pair_probability(ss):
    # fraction of frames each residue pair (i, j) is base-paired
    F, R = ss.shape
    M = np.zeros((R, R))
    for row in ss:
        for i, j in _frame_pairs(row):
            M[i, j] += 1.0; M[j, i] += 1.0
    return M / F


def plot_ss_pucker(pdbid):
    ss, pucker = load_ss_pucker(pdbid)
    M = base_pair_probability(ss)

    fig = plt.figure(figsize=(12, 5), constrained_layout=True)

    # 1) base-pair probability matrix
    ax1 = fig.add_subplot(1, 2, 1)
    im = ax1.imshow(np.ma.masked_where(M == 0, M), origin="lower", cmap=cmap,
                    vmin=0, vmax=1, interpolation="nearest")
    ax1.set_xlabel("Residue"); ax1.set_ylabel("Residue")
    panel_label(ax1, "a")
    fig.colorbar(im, ax=ax1, fraction=0.046, label="P(paired)")

    # 2) pseudorotation wheel (polar histogram of the pucker phase)
    ax2 = fig.add_subplot(1, 2, 2, projection="polar")
    phase = np.deg2rad(pucker.ravel())
    counts, edges = np.histogram(phase, bins=PUCKER_BINS, range=(0, 2 * np.pi))
    frac = counts / counts.sum()
    width = np.diff(edges); centers = edges[:-1] + width / 2
    ax2.bar(centers, frac, width=width, align="center",
            color=cmap(frac / frac.max()), edgecolor="white", linewidth=0.3)
    ax2.set_theta_zero_location("N"); ax2.set_theta_direction(-1)
    panel_label(ax2, "b")

    fig.suptitle(f"{pdbid}")
    plt.show()

In [ ]:
for pdbid in PDB_IDS:
    try:
        plot_ss_pucker(pdbid)
    except (FileNotFoundError, OSError, KeyError) as exc:
        print(f"{pdbid}: skipped ({exc})")

## Secondary structure & sugar pucker (set wide)


In [ ]:
# ---- configure ----
NORTH_MAX = 36.0   # deg; C3'-endo (North) window: phase < NORTH_MAX or > 360 - NORTH_MAX
# aggregates over PDB_IDS; reuses PUCKER_BINS, load_ss_pucker(), base_pair_probability() above
REL_SEP_GRID = 50   # common relative-separation grid for panel c

In [ ]:
from matplotlib.lines import Line2D


def collect_set_annotations(pdb_ids):
    phases, north_frac, paired_frac, sep_profiles, used = [], [], [], [], []
    for pid in pdb_ids:
        try:
            ss, pucker = load_ss_pucker(pid)
        except (FileNotFoundError, OSError, KeyError) as exc:
            print(f"{pid}: skipped ({exc})")
            continue
        ph = pucker.ravel()
        phases.append(ph)
        north_frac.append(float(((ph < NORTH_MAX) | (ph > 360 - NORTH_MAX)).mean()))
        paired_frac.append(float((ss != ".").mean()))
        M = base_pair_probability(ss)
        R = M.shape[0]
        sep_profiles.append(np.array([M[np.arange(R - k), np.arange(R - k) + k].mean()
                                      for k in range(1, R)]))
        used.append(pid)
    return dict(phases=np.concatenate(phases) if phases else np.array([]),
                north_frac=np.array(north_frac), paired_frac=np.array(paired_frac),
                sep_profiles=sep_profiles, used=used)


def plot_set_annotations(pdb_ids):
    d = collect_set_annotations(pdb_ids)
    n = len(d["used"])
    if n == 0:
        print("no systems with annotations"); return

    fig = plt.figure(figsize=(16, 4.5), constrained_layout=True)

    # (a) pooled pseudorotation wheel
    ax = fig.add_subplot(1, 3, 1, projection="polar")
    counts, edges = np.histogram(np.deg2rad(d["phases"]), bins=PUCKER_BINS, range=(0, 2 * np.pi))
    frac = counts / counts.sum(); width = np.diff(edges); centers = edges[:-1] + width / 2
    ax.bar(centers, frac, width=width, align="center",
           color=CUSTOM_COLORS["customhealth"], edgecolor="white", linewidth=0.3)
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
    panel_label(ax, "a")

    # (b) pairing probability vs absolute sequence separation
    ax = fig.add_subplot(1, 3, 2)
    maxlen = max(len(p) for p in d["sep_profiles"])
    P = np.full((n, maxlen), np.nan)
    for r, p in enumerate(d["sep_profiles"]):
        P[r, :len(p)] = p
    ks = np.arange(1, maxlen + 1)
    for r in range(n):
        ax.plot(ks, P[r], color=cmap(r / max(n - 1, 1)), alpha=0.35, lw=0.8)
    mean_h, = ax.plot(ks, np.nanmean(P, axis=0), color="black", lw=2.0)
    ax.set_xlabel("Sequence separation |i - j|"); ax.set_ylabel("P(paired)")
    sys_h = Line2D([0], [0], color=cmap(0.5), alpha=0.6, lw=0.8)   # per-system thin lines
    ax.legend([sys_h, mean_h], ["Single systems", "Set mean"])
    panel_label(ax, "b")

    # (c) pairing probability vs RELATIVE separation |i-j|/(n-1), each system
    #     interpolated onto a common grid so the set mean is over all systems.
    ax = fig.add_subplot(1, 3, 3)
    grid = np.linspace(0, 1, REL_SEP_GRID)
    Pr = np.full((n, REL_SEP_GRID), np.nan)
    for r, p in enumerate(d["sep_profiles"]):
        R = len(p) + 1
        rel = np.arange(1, R) / (R - 1)
        Pr[r] = np.interp(grid, rel, p, left=np.nan, right=np.nan)
    for r in range(n):
        ax.plot(grid, Pr[r], color=cmap(r / max(n - 1, 1)), alpha=0.35, lw=0.8)
    mean_h, = ax.plot(grid, np.nanmean(Pr, axis=0), color="black", lw=2.0)
    ax.set_xlabel("Relative separation |i - j| / (n - 1)"); ax.set_ylabel("P(paired)")
    sys_h = Line2D([0], [0], color=cmap(0.5), alpha=0.6, lw=0.8)   # per-system thin lines
    ax.legend([sys_h, mean_h], ["Single systems", "Set mean"])
    panel_label(ax, "c")
    plt.savefig(SAVE_DIR + 'base_sugar.pdf')
    plt.show()

In [ ]:
plot_set_annotations(PDB_IDS)

## Per-residue melting map



In [ ]:
# ---- configure ----
MELT_OBSERVABLE = "paired"   # "paired" (base pairing) or "north" (C3'-endo pucker)
# reads all temperature groups of H5_NAME; reuses _replica_labels() and NORTH_MAX
PDB_IDS_MELT = ['1xjr', '6lax', '7elq', '9obm']

In [ ]:
from matplotlib.patches import Arc


def melting_map(pdbid):
    # per-residue observable for every temperature group -> temps (T,), M (R, T)
    groups = _replica_labels(pdbid)
    temps = np.array([float(g) for g in groups])
    cols = []
    with h5py.File(os.path.join(DATA_ROOT, pdbid, H5_NAME.format(pdbid)), "r") as f:
        dom = f[list(f.keys())[0]]
        for g in groups:
            grp = dom[f"{g}/0"]
            if MELT_OBSERVABLE == "north":
                if "sugarPucker" not in grp:
                    raise KeyError("no sugarPucker (build with --rna-annotations)")
                ph = grp["sugarPucker"][:]
                cols.append(((ph < NORTH_MAX) | (ph > 360 - NORTH_MAX)).mean(axis=0))
            else:
                if "secondaryStructure" not in grp:
                    raise KeyError("no secondaryStructure (build with --rna-annotations)")
                ss = np.array([[c.decode() if isinstance(c, bytes) else c for c in row]
                               for row in grp["secondaryStructure"][:]])
                cols.append((ss != ".").mean(axis=0))
    return temps, np.array(cols).T


def frame0_pairs(pdbid):
    # base pairs in frame 0 of the lowest replica, plus residue count R
    with h5py.File(os.path.join(DATA_ROOT, pdbid, H5_NAME.format(pdbid)), "r") as f:
        dom = f[list(f.keys())[0]]
        groups = sorted((k for k in dom if isinstance(dom[k], h5py.Group) and k != "repex"),
                        key=float)
        row = dom[f"{groups[0]}/0"]["secondaryStructure"][0]
    row = [c.decode() if isinstance(c, bytes) else c for c in row]
    return _frame_pairs(row), len(row)


def draw_bow(ax, pairs, R):
    # arc (bow) diagram of base pairs, bulging left; residues 1..R on the y-axis
    maxspan = 1
    for i, j in pairs:
        span = abs(j - i)
        maxspan = max(maxspan, span)
        ax.add_patch(Arc((0, (i + j) / 2 + 1), width=span, height=span, angle=0,
                         theta1=90, theta2=270, color=CUSTOM_COLORS["customblue"], lw=1.0))
    ax.set_xlim(-maxspan / 2 * 1.15, maxspan * 0.03)
    ax.set_ylim(0.5, R + 0.5)
    ax.set_xticks([])
    ax.set_ylabel("Residue")


def plot_melting_grid(pdb_ids):
    label = "C3'-endo fraction" if MELT_OBSERVABLE == "north" else "P(paired)"
    n = len(pdb_ids)
    fig, axes = plt.subplots(1, 2 * n, figsize=(4.5 * n, 4.5), constrained_layout=True,
                             gridspec_kw={"width_ratios": [0.35, 1] * n})
    im = None
    for k, pdbid in enumerate(pdb_ids):
        ax_bow, ax_map = axes[2 * k], axes[2 * k + 1]
        try:
            temps, M = melting_map(pdbid)
            pairs, _ = frame0_pairs(pdbid)
        except (FileNotFoundError, OSError, KeyError) as exc:
            ax_bow.set_axis_off(); ax_map.set_axis_off()
            print(f"{pdbid}: skipped ({exc})"); continue
        R = M.shape[0]
        im = ax_map.pcolormesh(temps, np.arange(1, R + 1), M,
                               cmap=cmap, vmin=0, vmax=1, shading="nearest")
        ax_map.set_xlabel("Effective temperature [K]"); ax_map.set_title(f"{pdbid}")
        ax_map.set_ylim(0.5, R + 0.5)
        ax_map.tick_params(labelleft=False)            # residue axis shown on the bow (left)
        draw_bow(ax_bow, pairs, R)                      # shares the residue (y) range
    if im is not None:
        fig.colorbar(im, ax=axes.tolist(), label=label)
    plt.savefig(SAVE_DIR + 'melting.pdf')
    plt.show()

In [ ]:
plot_melting_grid(PDB_IDS_MELT)